# Data collection via Comdinheiro

The code below collects data on the composition of B3 indices (IBOV, IBX 100, SMLL and MCLX) and information on the stocks in these indices: price, volume, market_value and accounting data.

In [ ]:
# This notebook calls the Comdinheiro API, which requires a licensed account
# Set the environment variables below and keep the raw files in data/private/ before running it
import os

if not (os.environ.get('COMDINHEIRO_USERNAME') and os.environ.get('COMDINHEIRO_PASSWORD')):
    raise SystemExit('This notebook needs a Comdinheiro account: set COMDINHEIRO_USERNAME and COMDINHEIRO_PASSWORD. The analysis notebooks (03 to 05) run without it.')

In [ ]:
# Library imports

import pandas as pd
from datetime import date, datetime
from workadays import workdays as wd
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt
from  bizdays import Calendar, set_option
from time import sleep

cal = Calendar.load('ANBIMA')
set_option('mode', 'pandas')

In [ ]:
# Imports data already collected to make the analysis easier

path = '.'
factors_folder = 'factors'
data_folder = 'data'

companies = pd.read_csv(path + "/" + data_folder + "/" + 'index_composition.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
financials = pd.read_csv(path + "/" + data_folder + "/" + 'private/financials.csv.gz')
prices_df = pd.read_csv(path + "/" + data_folder + "/" + 'prices.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
market_value = pd.read_csv(path + "/" + data_folder + "/" + 'private/market_value.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
volume = pd.read_csv(path + "/" + data_folder + "/" + 'volume.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')
benchmarks = pd.read_csv(path + "/" + data_folder + "/" + 'benchmarks.csv.gz', index_col = 'Date', date_format='%Y-%m-%d')

# Index rebalancing dates
rebalance_dates = sorted(companies.index.unique())

# Backtest dates
ini_date = rebalance_dates[0]
end_date = date.today()

financials['statement_date'] = pd.to_datetime(financials['statement_date'], format='%Y-%m-%d')
financials['publication_date'] = pd.to_datetime(financials['publication_date'], format='%Y-%m-%d')
financials['collection_date'] = pd.to_datetime(financials['collection_date'], format='%Y-%m-%d')

# Data collection

The section below collects data separated into companies, prices and accounting information:
- Companies: we use the platform's stock screener to filter the companies that meet Greenblatt's criteria on several dates throughout the period analyzed.
- Prices: once we know which companies will be analyzed, we collect the daily closing prices of each one.
- Accounting information: we collect the accounting data needed to calculate ROC and EBIT/EV as Greenblatt asks in his book.

### Companies

Companies were collected based on the criteria below:
- We remove companies with average trading volume over the last 12 months below R$ 1,000,000.00.
- Companies in the MLCX (Mid Large Caps) and SMLL (Small Caps) indices; for companies with more than one ticker, those with the highest average liquidity over the last 12 months are selected.

In [ ]:
from brfactors.collection import get_index_composition

In [ ]:
from brfactors.collection import filter_indexes_comp

In [ ]:
# Creates a list with all rebalancing dates
dates = [cal.following(cal.getdate('first mon', year, month)) for year in range(2009, 2027) for month in [1,5,9]]

# Checks whether the date is a business day, if not, returns the next business day
dates = [ref_date for ref_date in dates if ref_date.date() < date.today()]

# Checks the dates already collected from the platform to avoid using up collections
collected_dates = [date.fromordinal(dt.toordinal()) for dt in companies.index.unique()]

# Cleans the list of dates to collect only the dates not yet collected
dates_to_collect = [ref_date for ref_date in dates if ref_date.date() not in collected_dates]

if dates_to_collect == []:
    print('The index composition is already up to date.')
    indexes = companies.copy()

else:
    # Creates a list with the dataframes of each date and concatenates them into a single dataframe
    mlcx_comp = pd.concat([get_index_composition(ref_date, 'MLCX') for ref_date in dates_to_collect])
    smll_comp = pd.concat([get_index_composition(ref_date, 'SMLL') for ref_date in dates_to_collect])
    ibov_comp = pd.concat([get_index_composition(ref_date, 'IBOV') for ref_date in dates_to_collect])
    ibrx_comp = pd.concat([get_index_composition(ref_date, 'IBRX') for ref_date in dates_to_collect])

    print('Index composition updated.')

        # Saves the dataframe to a file
    mlcx_comp['Index'] = 'MLCX'
    smll_comp['Index'] = 'SMLL'
    ibov_comp['Index'] = 'IBOV'
    ibrx_comp['Index'] = 'IBRX'

    # Creates a dataframe with all companies in the indices
    indexes = pd.concat([mlcx_comp, smll_comp, ibov_comp, ibrx_comp]).reset_index(drop = True)

# Creates a list with the companies' tickers
tickers = sorted(indexes['Ticker'].unique())

# Creates a string with the tickers separated by '%2B' to be used in the api url
tickers_str = tickers[0]
for ticker in tickers[1:]:
    tickers_str += '%2B' + ticker

indexes = filter_indexes_comp(indexes, tickers_str) if dates_to_collect != [] else indexes

indexes = pd.concat([companies, indexes.set_index('Date')]) if dates_to_collect != [] else indexes

# Saves the index companies to a file
indexes[['Ticker', 'Weight', 'Index', 'Sector']].to_csv(path + "/" + data_folder + "/" + 'index_composition.csv.gz')

### Benchmarks

We also collect the daily return of market benchmarks so we can compare performance. The period collected is January/2009 to May/2024.

In [ ]:
from brfactors.collection import get_benchmarks

In [ ]:
last_date = benchmarks.index[-1]
if last_date.date() < wd.workdays(date.today(), -1):
    dt_start = last_date
    dt_end = wd.workdays(date.today(), -1)

    # Collects the updated data from the last collected day
    benchmarks_new_data = get_benchmarks(dt_start, dt_end)

    # Concatenates the old data with the new data
    benchmarks_updated = pd.concat([benchmarks, benchmarks_new_data])

    # Saves the dataframe to a file
    benchmarks_updated.to_csv(path + "/" + data_folder + "/" + 'benchmarks.csv.gz')

    benchmarks = benchmarks_updated.copy()

    print('Benchmarks updated.')
else:
    print('Benchmarks are already up to date.')

### Prices

Now that we have the companies that will take part in the analysis, we collect the daily closing prices of each one. The period collected is January/2007 to May/2024.

In [ ]:
from brfactors.collection import get_prices

In [ ]:
new_tickers = [tik for tik in tickers if tik not in prices_df.columns]

if len(new_tickers) == 0:
    print('No stock added')

elif len(new_tickers) == 1:
    new_price = get_prices(new_tickers[0], date(2004,1,1), prices_df.index[-1])
    prices_df = prices_df.merge(new_price, right_index=True, left_index=True, how = 'left')

    print('New stock added.')
    print(new_tickers)

else:
    new_tickers_str = new_tickers[0]
    for ticker in new_tickers[1:]:
        new_tickers_str += '%2B' + ticker

    new_prices = get_prices(new_tickers_str, date(2004,1,1), prices_df.index[-1])
    prices_df = prices_df.merge(new_prices, right_index=True, left_index=True, how = 'left')

    print('New stocks added.')
    print(new_tickers)

prices_df = prices_df[tickers]

In [ ]:
last_date = prices_df.index[-1]
if last_date.date() < wd.workdays(date.today(), -1):
    dt_start = wd.workdays(last_date, 1)
    dt_end = wd.workdays(date.today(), -1)

    # Collects the updated data from the last collected day
    updated_prices = get_prices(tickers_str, date(2004,1,1), dt_end)

    # Saves the dataframe to a file
    updated_prices.to_csv(path + "/" + data_folder + "/" + 'prices.csv.gz')

    prices_df = updated_prices.copy()

    print('Prices updated.')
else:
    print('Prices are already up to date.')

### Market Value

Now that we have the companies that will take part in the analysis, we collect the companies' daily market value. The period collected is January/2008 to May/2024.

In [ ]:
from brfactors.collection import get_market_value

In [ ]:
new_tickers = [tik for tik in tickers if tik not in market_value.columns]

if len(new_tickers) == 0:
    print('No stock added')

elif len(new_tickers) == 1:
    new_price = get_market_value(new_tickers[0], date(2008,1,2), market_value.index[-1])
    market_value = market_value.merge(new_price, right_index=True, left_index=True, how = 'left')

    print('New stock added.')
    print(new_tickers)

else:
    for subset in np.array_split(new_tickers, int(len(new_tickers) / 5) + (len(new_tickers) % 5 > 0)):
        new_tickers_str = subset[0]
        for ticker in subset[1:]:
            new_tickers_str += '%2B' + ticker

        new_mv = get_market_value(new_tickers_str, date(2008,1,2), market_value.index[-1])
        market_value = market_value.merge(new_mv, right_index=True, left_index=True, how = 'left')

        print('New stocks added: ' + str(subset))

market_value = market_value[tickers]

In [ ]:
last_date = market_value.index[-1]

if last_date.date() < wd.workdays(date.today(), -1):
    dt_start = wd.workdays(last_date, 1)
    dt_end = wd.workdays(date.today(), -1)

    # Collects the updated data from the last collected day
    updated_mv = get_market_value(tickers_str, dt_start, dt_end)

    # Concatenates the old data with the new data
    mv_updated = pd.concat([market_value, updated_mv])

    # Saves the dataframe to a file
    mv_updated.to_csv(path + "/" + data_folder + "/" + 'private/market_value.csv.gz')

    market_value = mv_updated.copy()

    print('Market values updated.')
else:
    print('Market values are already up to date.')

### Daily Volume

Now that we have the companies that will take part in the analysis, we collect the stocks' trading volume. The period collected starts in January/2008.

In [ ]:
from brfactors.collection import get_volume

In [ ]:
new_tickers = [tik for tik in tickers if tik not in volume.columns]

if len(new_tickers) == 0:
    print('No stock added')

elif len(new_tickers) == 1:
    new_price = get_volume(new_tickers[0], date(2008,1,2), volume.index[-1])
    volume = volume.merge(new_price, right_index=True, left_index=True, how = 'left')

    print('New stock added.')
    print(new_tickers)

else:
    for subset in tqdm(np.array_split(new_tickers, int(len(new_tickers) / 5) + (len(new_tickers) % 5 > 0))):
        new_tickers_str = subset[0]
        for ticker in subset[1:]:
            new_tickers_str += '%2B' + ticker

        new_volume = get_volume(new_tickers_str, date(2008,1,2), volume.index[-1])
        volume = volume.merge(new_volume, right_index=True, left_index=True, how = 'left')

        print('New stocks added: ' + str(subset))

volume = volume[tickers]

In [ ]:
last_date = volume.index[-1]

if last_date.date() < wd.workdays(date.today(), -1):
    dt_start = wd.workdays(last_date, 1)
    dt_end = wd.workdays(date.today(), -1)

    # Collects the updated data from the last collected day
    updated_volume = get_volume(tickers_str, dt_start, dt_end)

    # Concatenates the old data with the new data
    volume_updated = pd.concat([volume, updated_volume])

    # Saves the dataframe to a file
    volume_updated.to_csv(path + "/" + data_folder + "/" + 'volume.csv.gz')

    volume = volume_updated.copy()

    print('Volumes updated.')
else:
    print('Volumes are already up to date.')

### Accounting information

We use the companies and the tickers_str variable to collect the accounting information for calculating ROC and EBIT/EV. The information collected is:
- Total Assets;
- Current Assets;
- Current Liabilities;
- Short-Term Loans;
- Intangibles;
- EBIT: Earnings Before Interest and Taxes;
- Net Debt;
- Publication date of the financial statements.

In [ ]:
from brfactors.collection import get_financial_statements

In [ ]:
closing_dates = [cal.getdate('last day', y, m) for y in range(2008, 2027) for m in [3,6,9,12]][1:-3]
dt_analysis = cal.preceding(date.today())

financials_new = pd.concat([get_financial_statements(dt_analysis, dt_ref, tickers_str) for dt_ref in tqdm(closing_dates)])

In [ ]:
new_columns = ['ticker', 'statement_date', 'publication_date', 
                 'total_assets', 'current_assets', 'cash', 'accounts_receivable', 'inventories', 'recoverable_taxes', 'other_current_assets', 
                 'non_current_assets', 'lt_receivables', 'lt_investments', 'ppe', 'intangibles', 
                 'current_liabilities', 'suppliers_st', 'st_debt', 'labor_obligations', 'dividends_payable', 'tax_obligations', 
                 'non_current_liabilities', 'suppliers_lt', 'lt_debt', 'equity', 
                 'net_revenue', 'gross_profit', 'ebit', 'ebitda', 'pretax_income', 'net_income', 
                 'depreciation_amortization', 'fco', 'fci', 'fcf', 'cash_change'
                 ]

financials_new.columns = new_columns
financials_new = financials_new.sort_values(by = ['ticker', 'statement_date', 'publication_date'], ascending = [True, True, True])
financials_new = financials_new.drop_duplicates(subset = new_columns[0:2] + new_columns[3:], keep = 'first').copy()

financials_new.loc[(financials_new['publication_date'].isna()) & (financials_new['ticker'] == 'AUTM3'), 'publication_date'] = date(2012, 7, 25)
financials_new['ticker'] = financials_new['ticker'].str.replace('TIET11un', 'TIET11')
financials_new['collection_date'] = date.today()

In [ ]:
# Vertically concatenates the dataframe with the new information to the old one
concat = pd.concat([financials, financials_new]).reset_index(drop = True)

# Removes the rows with none of the values requested from the API
concat = concat.loc[concat.replace(0,np.nan).isna().sum(axis = 1) < (len(concat.columns) - 4)]

# Removes duplicated rows considering only the accounting data, excluding per-share data and the request date
concat = concat.drop_duplicates(subset = new_columns, keep = 'first')

# Converts date columns to datetime.date
concat[['statement_date', 'publication_date', 'collection_date']] = concat[['statement_date', 'publication_date', 'collection_date']].apply(pd.to_datetime)

# Sorts by ticker, fiscal date and request date
concat = concat.sort_values(by = ['ticker', 'statement_date', 'publication_date', 'collection_date'], ascending = [True, True, True, True])
concat = concat.dropna(subset = 'publication_date').reset_index(drop = True)

In [ ]:
# Saves the dataframe to a file
concat.to_csv(path + "/" + data_folder + "/" + 'private/financials.csv.gz', index = False)